In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hashlib
import json
import time
from collections.abc import Callable
from dataclasses import dataclass, field

# Concept 4: Permissioned chains — a chain where the guest list matters

Companion to notebooks 1–3. Quick recap of the story so far, because this
notebook is about to change nearly everything you've gotten used to:

| Notebook | Question it answers | Mechanism |
| --- | --- | --- |
| 1. Mining | How do we make history tamper-evident, and rewriting *expensive*? | Hash-chained blocks + proof-of-work |
| 2. Proof of Stake | Who gets to propose the next block, without burning a small country's electricity bill? | Stake-weighted lottery + slashing |
| 3. Merkle trees | How do we commit to a *huge* batch of data cheaply, and prove one item belongs without exposing the rest? | Binary hash tree + O(log N) proofs |
| **4. This notebook** | What if we *don't* want just anyone to join, there's no token to mine or stake, and several orgs that don't fully trust each other still need to agree on one shared ledger? | Membership + endorsement + ordering + commit-time validation |

Notebooks 1–3 were all, deep down, the same shape: **one open network,
one rule for "who may write next"** (find a nonce / win the stake
lottery), and **one validator loop** that re-checks the *whole chain*
from genesis whenever asked. Fun fact: nobody ever had to prove who they
*were* — a hash doesn't care about your identity, only your CPU cycles
or your stake.

That's exactly wrong for, say, three banks running a shared settlement
ledger. They don't want:

- randoms joining the network,
- a proof-of-work energy bill or a speculative staking token,
- or a system where "51% of the CPU" or "51% of the stake" decides the
  truth, when the actual participants are a known, small, named list of
  institutions who'd very much like a court to be able to point at who
  signed what.

This notebook builds a **permissioned (consortium) chain**, modelled
loosely on **Hyperledger Fabric**. The core shift: instead of "whoever
wins gets to propose, then everyone re-verifies," Fabric-style chains
run **execute → endorse → order → validate** — multiple named
organizations *independently execute and agree* on a transaction's
outcome *before* it ever gets sequenced into a block.

**What we build here (toy model):**

1. A `MembershipService` — gatekeeping who's even allowed to hold a
   peer identity (no entry, no participation, full stop)
2. `transfer_chaincode` — the actual "smart contract" business logic
3. `Peer.endorse()` — named orgs *execute* a proposed transaction and
   sign off on the result, before anyone tries to order it
4. `OrderingService` — a dumb, opinion-free sequencer (it does **not**
   validate anything)
5. `PermissionedBlockchain.validate_and_commit()` — commit-time checks:
   endorsement policy satisfied? endorsers agree? and **MVCC**
   (multi-version concurrency control) — is the world state still what
   the endorsers thought it was?
6. `propose_with_retry()` — the client-side retry loop for when two
   transactions collide over the same piece of state

> **Honest scope:** this is a single-process simulation of Fabric's
> *shape*, not Fabric itself. No real PKI/X.509 identities, no chaincode
> containers, no channels or private data collections, no Raft/BFT
> consensus inside the ordering service, and no network partitions. What
> *is* faithful: the execute-before-order pipeline, and the reason MVCC
> conflict detection exists at all. Treat this the way you'd treat a
> floor plan — accurate about the rooms and doors, not architecturally
> load-bearing.

## Gate #1: Membership — permission to even show up

In notebooks 1–3, "who can participate" was answered by physics or
economics: anyone with a CPU can mine (PoW), anyone can lock up tokens
and enter the lottery (PoS). There was no admissions office.

Here there is one. `MembershipService` is a deliberately tiny stand-in
for what Fabric calls an **MSP (Membership Service Provider)** — in real
life backed by X.509 certificates issued by a CA that each consortium
member trusts. No entry in the registry, no participation:
`Peer.endorse()` checks `is_registered()` before doing anything else,
and throws if you're not on the list.

This is the permissioned chain's first and most fundamental departure
from notebooks 1–3: **identity is a precondition, not an emergent
property of effort or stake.**

In [3]:
def sha256(data: str) -> str:
    return hashlib.sha256(data.encode()).hexdigest()


class MembershipService:
    def __init__(self) -> None:
        self.registered_orgs: set[str] = set()

    def register(self, org_name: str) -> None:
        print(f"  [MembershipService] Registered org: {org_name}")
        self.registered_orgs.add(org_name)

    def is_registered(self, org_name: str) -> bool:
        return org_name in self.registered_orgs

## The ledger's state: a versioned key-value world, not just a list of blocks

Notebooks 1–3 derived "current state" by replaying the chain (or just
mutated a `Validator.stake` field directly). That's fine for a toy with
one writer at a time.

Here, multiple orgs can *concurrently* propose transactions against the
same shared state, so we need to answer a sharper question than "what's
the current balance?" — we need to ask **"what version of this key was
I looking at when I computed my answer?"**

`WorldState` tracks two things per key:

| Field | Role |
| --- | --- |
| `balances` | The actual current values (a normal key-value store) |
| `versions` | A counter, bumped every time a key is written |

Hang onto `version_of()` — it's the hook the whole optimistic-concurrency
story further down is built on. `ExecutionResult` is what a chaincode
run produces: a `write_set` (what it wants to change) and
`read_versions` (what versions it *assumed* were current when it
computed that write set). Neither is applied to `WorldState` yet — this
is a **proposal**, not a commit.

In [4]:
@dataclass
class WorldState:
    balances: dict[str, float] = field(default_factory=dict)
    versions: dict[str, int] = field(default_factory=dict)

    def version_of(self, key: str) -> int:
        return self.versions.get(key, 0)


@dataclass
class ExecutionResult:
    write_set: dict[str, float]
    read_versions: dict[str, int]

## Chaincode: the business logic, run speculatively

"Chaincode" is Fabric's word for what other platforms call a **smart
contract**. `transfer_chaincode` is about as simple as one gets: check
a balance, compute a new pair of balances.

The important part isn't the arithmetic — it's *when* this runs.
Executing this function does **not** touch the real ledger. It runs
against a state snapshot, tentatively, and returns an `ExecutionResult`
that some `Peer` will sign off on. Two different peers, running this
same function against the same state, had better get the *same*
`write_set` — that agreement is what "endorsement" is actually checking
in a moment.

In [5]:
def transfer_chaincode(state: WorldState, frm: str, to: str, amount: float) -> ExecutionResult:
    if state.balances.get(frm, 0) < amount:
        raise ValueError(f"Insufficient funds: {frm} has {state.balances.get(frm, 0)}, needs {amount}")
    return ExecutionResult(
        write_set={frm: state.balances.get(frm, 0) - amount, to: state.balances.get(to, 0) + amount},
        read_versions={frm: state.version_of(frm), to: state.version_of(to)},
    )

## Execute → Endorse → Order → Validate

This is the shape swap. Compare it to notebooks 1–3's loop of
*(one node proposes) → (everyone re-checks a completed block)*:

| Phase | Who does it | What happens |
| --- | --- | --- |
| **Execute** | Each endorsing peer, independently | Run the chaincode against *their own* view of state; get a candidate `write_set` |
| **Endorse** | Same peers | Sign off — here, just record a hash of the `write_set` — *without* touching the shared ledger |
| **Order** | `OrderingService` | Sequence proposals into a canonical order. No business logic, no opinion on correctness — just a queue |
| **Validate** | `PermissionedBlockchain.validate_and_commit()` | *Now* check: enough endorsers? do they agree? is the state they read still current? |

Why bother separating execute from order, when notebooks 1–3 just did
"propose, mine/sign, done"? Because here **multiple mutually
distrusting organizations** are involved. You can't let one party
compute the result and have everyone else take their word for it — so
you make several of them compute it *independently* and check they
land on the same answer, and you do that check **before** wasting
everyone's time sequencing it.

`Endorsement` intentionally carries only a **hash** of the write set,
not the write set itself — the full result travels alongside in
`ExecutionResult` for this toy, but in real Fabric this separation is
what lets the ordering service sequence transactions **without ever
seeing their contents** (useful when channels carry sensitive data).

`OrderingService.order()` is almost insultingly simple on purpose: it
gets paid to put things in a queue, and has zero opinion on whether
what's in the queue is honest. In real Fabric this is itself a cluster
running Raft (or historically, Kafka) — consensus on *ordering*, kept
completely separate from consensus on *correctness*.

In [6]:
@dataclass
class Endorsement:
    peer_name: str
    result_hash: str


@dataclass
class Peer:
    name: str
    membership: MembershipService

    def endorse(self, chaincode: Callable, state: WorldState, *args) -> tuple[Endorsement, ExecutionResult]:
        if not self.membership.is_registered(self.name):
            raise PermissionError(f"{self.name} is not a registered participant.")
        result = chaincode(state, *args)
        result_hash = sha256(json.dumps(result.write_set, sort_keys=True))
        return Endorsement(peer_name=self.name, result_hash=result_hash), result


@dataclass
class ProposedTransaction:
    description: str
    endorsements: list[Endorsement]
    execution_result: ExecutionResult


class OrderingService:
    def __init__(self, name: str) -> None:
        self.name = name

    def order(self, proposals: list[ProposedTransaction]) -> list[ProposedTransaction]:
        print(f"  [{self.name}] Sequencing {len(proposals)} proposal(s).")
        return list(proposals)

## Finally, the ledger itself

One more shape change before the `Block` and `PermissionedBlockchain`
classes below: there's no `nonce` (no mining, notebook 1) and no
`proposer`/`stake` (no lottery, notebook 2). Instead there's an
`orderer` field — a name, not a competitor in any race, just a record
of which ordering service sequenced this block. All the interesting
legitimacy checks happen in `validate_and_commit()`, not in the block
itself.

In [7]:
class Block:
    def __init__(self, index: int, data: str, previous_hash: str, orderer: str) -> None:
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.orderer = orderer
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        block_contents = json.dumps(
            {"index": self.index, "timestamp": self.timestamp, "data": self.data,
             "previous_hash": self.previous_hash, "orderer": self.orderer},
            sort_keys=True,
        )
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def __repr__(self) -> str:
        return f"Block #{self.index}: {self.data}\n"

### A noticeably boring `Block`

Notice what's *missing* compared to earlier notebooks: no proof-of-work
target, no proposer eligibility field. `hash` and `previous_hash` are
still here — tamper-evidence within a block and links between blocks
haven't gone anywhere — but they're no longer the main event. The main
event, `validate_and_commit()` below, happens once **per transaction**,
*before* a block is even created, not by re-scanning the whole chain
after the fact the way `is_valid()` did in notebooks 1 and 2.

In [8]:
class PermissionedBlockchain:
    def __init__(self, required_endorsers: set[str], min_endorsements: int, orderer_name: str) -> None:
        self.required_endorsers = required_endorsers
        self.min_endorsements = min_endorsements
        self.orderer_name = orderer_name
        self.chain: list[Block] = [Block(0, "Genesis Block", "0" * 64, orderer_name)]
        self.state = WorldState()

    def validate_and_commit(self, tx: ProposedTransaction) -> bool:
        endorser_names = {e.peer_name for e in tx.endorsements}
        if not endorser_names.issuperset(self.required_endorsers):
            print("    REJECTED: missing required endorsers")
            return False
        if len(tx.endorsements) < self.min_endorsements:
            print("    REJECTED: not enough endorsements")
            return False
        if len({e.result_hash for e in tx.endorsements}) > 1:
            print("    REJECTED: endorsers disagree")
            return False
        for key, read_version in tx.execution_result.read_versions.items():
            current_version = self.state.version_of(key)
            if current_version != read_version:
                print(f"    REJECTED: MVCC CONFLICT on '{key}' -- endorsed against "
                      f"version {read_version}, current is {current_version}")
                return False
        for key, new_value in tx.execution_result.write_set.items():
            self.state.balances[key] = new_value
            self.state.versions[key] = self.state.version_of(key) + 1
        record = json.dumps({"description": tx.description, "write_set": tx.execution_result.write_set}, sort_keys=True)
        previous_block = self.chain[-1]
        new_block = Block(len(self.chain), record, previous_block.hash, self.orderer_name)
        self.chain.append(new_block)
        print(f"    COMMITTED as Block #{new_block.index}")
        return True

## `validate_and_commit()` — four gates, in order

This is where "permissioned" earns its keep. For every proposed
transaction:

1. **Endorsement policy** — did *all* `required_endorsers` actually sign
   off? (Here: both banks, always — a strict "everyone must agree"
   policy. Real Fabric policies can be richer, e.g. "any 2 of 3".)
2. **Quorum** — at least `min_endorsements` signatures, full stop.
3. **Agreement** — do all endorsers' `result_hash`es match? If two
   honest-looking endorsers computed *different* write sets, something
   is wrong (bug, malicious peer, or stale state) — reject rather than
   guess who's right.
4. **MVCC conflict check** — for every key the transaction *read*, is
   `state.version_of(key)` still what the endorsers assumed? If someone
   else's transaction already landed and bumped that version, this
   proposal is stale — reject it, no matter how honest it was.

Only after all four pass does the write set actually land in
`WorldState`, versions get bumped, and a new `Block` gets appended.

> **Honest scope gap, named on purpose:** unlike notebooks 1 and 2, this
> `PermissionedBlockchain` has no `is_valid()` that re-walks the whole
> chain checking every `hash`/`previous_hash` link from genesis. Each
> block *could* still be re-verified that way — the fields are all
> there — we just don't wire it up here, because the interesting new
> idea in this notebook is commit-time transaction validation, not
> chain replay. Bolting that check back on would be a straightforward
> exercise, not a gap in the idea.

### Then: what happens when two transactions collide?

Check #4 above is where **optimistic concurrency control (OCC)** lives.
"Optimistic" because nobody locks anything up front — every peer just
executes hopefully, assuming nothing else will change the state before
their transaction lands. Most of the time that's true, and it's free
efficiency. When it's *not* true, `validate_and_commit()` catches the
stale read at commit time and rejects — and `propose_with_retry()`
below is the client-side response: shrug, re-run `execute` against
whatever the state actually is *now*, and try again.

In [9]:
def propose_with_retry(
    description: str,
    peers: list[Peer],
    chaincode: Callable,
    chain: PermissionedBlockchain,
    orderer: OrderingService,
    args: tuple,
    max_attempts: int = 3,
) -> bool:
    """The client-side retry loop -- what a real Fabric SDK does
    automatically. On an MVCC conflict, don't give up: re-run EXECUTE
    against whatever the CURRENT state now is, and try again. This is
    the 'optimistic' half of optimistic concurrency control -- pay the
    retry cost only on an actual collision, not on every transaction.

    Args:
        description: Label for this transaction.
        peers: Endorsing peers to use.
        chaincode: The smart contract function to run.
        chain: The ledger to submit to.
        orderer: The ordering service to sequence through.
        args: Arguments to pass to the chaincode.
        max_attempts: How many times to retry before giving up.

    Returns:
        True if eventually committed, False if all attempts were exhausted.
    """
    for attempt in range(1, max_attempts + 1):
        print(f"  Attempt {attempt}: {description} (fresh EXECUTE against current state)")
        endorsements: list[Endorsement] = []
        execution_result: ExecutionResult | None = None
        for peer in peers:
            endorsement, result = peer.endorse(chaincode, chain.state, *args)
            endorsements.append(endorsement)
            execution_result = result
        tx = ProposedTransaction(description, endorsements, execution_result)
        ordered = orderer.order([tx])
        if chain.validate_and_commit(ordered[0]):
            return True
        print("    -> stale, retrying...\n")
    print(f"  Gave up after {max_attempts} attempts.")
    return False


## Demo: two banks, one shared ledger, zero shared trust

The scenario: `Mastercard-Bank-A` and `Mastercard-Bank-B` jointly run a
settlement ledger. Neither trusts the other's arithmetic, so the
endorsement policy requires **both** to independently compute and agree
on every transaction before it commits. Nobody is mining. Nobody is
staking. Everybody is, presumably, billing hours.

In [10]:
print("=== Setup ===\n")
membership = MembershipService()
membership.register("Mastercard-Bank-A")
membership.register("Mastercard-Bank-B")
peer_a = Peer("Mastercard-Bank-A", membership)
peer_b = Peer("Mastercard-Bank-B", membership)
peers = [peer_a, peer_b]

chain = PermissionedBlockchain(
    required_endorsers={"Mastercard-Bank-A", "Mastercard-Bank-B"},
    min_endorsements=2,
    orderer_name="Shared-Ordering-Service",
)
orderer = OrderingService("Shared-Ordering-Service")
chain.state.balances = {"Alice-Account": 1000.0, "Bob-Account": 500.0, "Carol-Account": 0.0}
print(f"  Starting balances: {chain.state.balances}\n")

=== Setup ===

  [MembershipService] Registered org: Mastercard-Bank-A
  [MembershipService] Registered org: Mastercard-Bank-B
  Starting balances: {'Alice-Account': 1000.0, 'Bob-Account': 500.0, 'Carol-Account': 0.0}



### Here comes the race condition

Two clients submit **concurrently**: Alice pays Bob $200, and — before
that lands anywhere — Alice pays Carol $150. Both get endorsed against
the *same* starting version of `Alice-Account` (version 0), because
neither peer knew about the other proposal yet. Only one of these can
possibly be valid once both try to land, since together they'd overdraw
Alice. Watch `validate_and_commit()` sort it out by version number, not
by which one "seems more honest."

In [11]:
print("=== Two clients endorse CONCURRENTLY, both against Alice-Account version 0 ===\n")
e_a1, res1 = peer_a.endorse(transfer_chaincode, chain.state, "Alice-Account", "Bob-Account", 200.0)
e_b1, _ = peer_b.endorse(transfer_chaincode, chain.state, "Alice-Account", "Bob-Account", 200.0)
tx1 = ProposedTransaction("Alice pays Bob $200", [e_a1, e_b1], res1)

e_a2, res2 = peer_a.endorse(transfer_chaincode, chain.state, "Alice-Account", "Carol-Account", 150.0)
e_b2, _ = peer_b.endorse(transfer_chaincode, chain.state, "Alice-Account", "Carol-Account", 150.0)
tx2_stale_attempt = ProposedTransaction("Alice pays Carol $150", [e_a2, e_b2], res2)
print("  Both endorsed. Neither peer knew about the other proposal.\n")

=== Two clients endorse CONCURRENTLY, both against Alice-Account version 0 ===

  Both endorsed. Neither peer knew about the other proposal.



In [12]:
print("=== ORDER + VALIDATE: Tx1 happens to get sequenced first ===\n")
ordered1 = orderer.order([tx1])
chain.validate_and_commit(ordered1[0])

=== ORDER + VALIDATE: Tx1 happens to get sequenced first ===

  [Shared-Ordering-Service] Sequencing 1 proposal(s).
    COMMITTED as Block #1


True

In [13]:
print("\n=== Tx2's ORIGINAL (now-stale) endorsement finally reaches Validate ===\n")
success = chain.validate_and_commit(tx2_stale_attempt)
print(f"  First attempt succeeded? {success}\n")


=== Tx2's ORIGINAL (now-stale) endorsement finally reaches Validate ===

    REJECTED: MVCC CONFLICT on 'Alice-Account' -- endorsed against version 0, current is 1
  First attempt succeeded? False



In [14]:
print("=== Client retries automatically, re-executing against CURRENT state ===\n")
success = propose_with_retry(
    "Alice pays Carol $150", peers, transfer_chaincode, chain, orderer,
    ("Alice-Account", "Carol-Account", 150.0),
)
print(f"\n  Eventually succeeded? {success}\n")

=== Client retries automatically, re-executing against CURRENT state ===

  Attempt 1: Alice pays Carol $150 (fresh EXECUTE against current state)
  [Shared-Ordering-Service] Sequencing 1 proposal(s).
    COMMITTED as Block #2

  Eventually succeeded? True



In [15]:
print("=== Final state ===")
print(f"  Balances: {chain.state.balances}")
print(f"  Versions: {chain.state.versions}")
print(f"  Chain length: {len(chain.chain)} blocks")

=== Final state ===
  Balances: {'Alice-Account': 650.0, 'Bob-Account': 700.0, 'Carol-Account': 150.0}
  Versions: {'Alice-Account': 2, 'Bob-Account': 1, 'Carol-Account': 1}
  Chain length: 3 blocks


---

## Takeaways

1. **Membership is a precondition here, not an emergent property** —
   compare to notebooks 1–2, where "who can participate" fell out of
   physics (CPU) or economics (stake).
2. **Execute happens before order** — several named orgs independently
   run the chaincode and must *agree*, before anything gets sequenced.
   Notebooks 1–3 only ever had one party compute a result at a time.
3. **The ordering service is deliberately dumb** — sequencing and
   correctness-checking are different jobs, done by different
   components, so a single dishonest or broken orderer can't forge
   agreement it never actually confirmed.
4. **MVCC replaces "51% of something" as the anti-cheating mechanism** —
   there's no hash puzzle and no stake at risk; conflicting transactions
   are caught by comparing state *versions* at commit time.
5. **Optimistic concurrency control** trades a small chance of wasted
   work (a rejected, retried transaction) for never having to lock
   shared state up front — cheap when collisions are rare.

### The whole series, side by side

| | 1. PoW | 2. PoS | 3. Merkle trees | 4. Permissioned |
| --- | --- | --- | --- | --- |
| Who can join | Anyone with a CPU | Anyone who stakes | N/A (a data structure, not a network) | Only registered members |
| Right to write | Win the hash race | Win the stake lottery | — | Endorsement policy satisfied |
| What stops cheating | Re-mining is expensive | Slashing destroys stake | Collision-resistant hashing | MVCC + multi-party agreement |
| Validation happens | Re-scan whole chain | Re-scan whole chain | Recompute path to root | Per-transaction, at commit time |
| Real-world cousin | Bitcoin | Ethereum (post-Merge) | Certificate Transparency, tx batches | Hyperledger Fabric, R3 Corda |

**Still out of scope (on purpose), if you want to keep digging:** real
PKI-backed membership (X.509, CAs, revocation), chaincode running in
actual isolated containers, **channels** and **private data
collections** for confidentiality between subsets of members, real
consensus *inside* the ordering service (Raft, BFT), and what happens
when an endorsing peer is not just slow but actively lying (Byzantine
behaviour — this notebook's "endorsers disagree" check catches honest
disagreement, not a coordinated lie).

Four notebooks, four different answers to the same underlying question:
**how do a bunch of people who don't fully trust each other agree on
one shared history?** Burn electricity, put money at risk, hash your way
to a cheap membership check, or — as it turns out most real enterprises
actually do — just make everyone show ID at the door.